# 🎙️ Speech-to-Text API — Google Colab Setup & Test

**Stack:** faster-whisper · WhisperX (optional alignment) · pyannote.audio · FastAPI · Redis · RQ

> **Before you start:** Go to **Runtime → Change runtime type** and select **T4 GPU**.

## 1 · Environment Detection & GPU Verification

In [ ]:
import sys, os

# Detect Colab
IN_COLAB = "google.colab" in sys.modules
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

print(f"Running in Colab : {IN_COLAB}")
print(f"Python           : {sys.version}")

# GPU check
import subprocess
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode == 0:
    print("\n--- nvidia-smi ---")
    # Print only the first 15 lines to keep output compact
    print("\n".join(result.stdout.splitlines()[:15]))
else:
    print("⚠️  nvidia-smi not found — make sure you selected a GPU runtime!")

# PyTorch CUDA info (torch may not be installed yet — that is fine)
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    print(f"\ntorch {torch.__version__}  |  CUDA available: {cuda_ok}")
    if cuda_ok:
        print(f"GPU  : {torch.cuda.get_device_name(0)}")
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"VRAM : {vram_gb:.1f} GB")
        print(f"CUDA : {torch.version.cuda}")
        # Recommend compute_type
        if vram_gb >= 10:
            print("✅  Recommended compute_type: float16  (batch_size: 16)")
        else:
            print("⚠️  Low VRAM — use compute_type: int8  and batch_size: 8")
except ImportError:
    print("torch not installed yet — will be installed in the next step")

## 2 · System Dependencies

In [ ]:
%%bash
# Install system packages (quiet)
apt-get update -qq
apt-get install -y -qq ffmpeg libsndfile1 redis-server

# Start Redis (background, quiet)
redis-server --daemonize yes --loglevel warning
sleep 1
redis-cli ping && echo "✅ Redis is up" || echo "❌ Redis failed"

echo "✅ System packages installed"

## 3 · Python Packages — Correct Install Order

**Strategy for Colab:**
- ✅ **Keep** Colab's pre-installed `torch`, `numpy`, `torchaudio` — do NOT reinstall or downgrade them
- ✅ **Install `whisperx --no-deps`** so pip cannot touch our torch version
- ✅ **Install whisperx runtime deps** explicitly at versions that don't conflict with Colab
- ✅ **Use `v3.8.5`** (latest real tag, requires `torch>=2.8`, `numpy>=2.1` — both satisfied by Colab)

> Colab (April 2026) ships `torch 2.10.0+cu128`, `numpy 2.x`, `torchaudio 2.7.0`.  
> WhisperX `v3.8.5` requires `torch~=2.8.0` — Colab's `2.10.0` satisfies this.

In [ ]:
%%bash
# NOTE: Do NOT use set -e here — we allow pip conflict warnings (non-zero exit on warns)
# but still want to catch real errors. We check each critical step manually.

echo "=== Step 1: Upgrade pip / build tools ==="
pip install --quiet --upgrade pip setuptools wheel

echo ""
echo "=== Step 2: Clone the repo (skip if already present) ==="
if [ ! -d /content/speech2text ]; then
    git clone --quiet --depth 1 \
        https://github.com/Sanjay9745/whisperx-speech2text.git \
        /content/speech2text
    echo "Cloned to /content/speech2text"
else
    echo "Repo already at /content/speech2text — skipping clone"
fi

echo ""
echo "=== Step 3: Install WhisperX runtime deps (Colab-compatible versions) ==="
# We install these BEFORE whisperx itself so whisperx --no-deps can skip them.
# Versions chosen to NOT conflict with Colab's pre-installed torch 2.10 / numpy 2.x.
pip install --quiet \
    "ctranslate2>=4.5.0" \
    "faster-whisper>=1.2.0" \
    "nltk>=3.9.1" \
    "omegaconf>=2.3.0" \
    "pyannote.audio>=4.0.0" \
    "pyannote.core>=5.0.0" \
    "pyannote.pipeline>=3.0.1"

echo ""
echo "=== Step 4: Install application requirements ==="
# torch / torchaudio / numpy already installed by Colab — pip will skip them.
# We override the pins from requirements.txt that are too old for current Colab.
pip install --quiet \
    "fastapi>=0.124.1" \
    "uvicorn[standard]>=0.34.0" \
    "starlette>=0.49.1" \
    "python-multipart>=0.0.18" \
    "huggingface-hub>=0.33.5,<1.0.0" \
    "httpx>=0.28.1" \
    "redis>=5.2.0" \
    "rq>=2.0.0" \
    "pydub>=0.25.1" \
    "soundfile>=0.12.1" \
    "librosa>=0.10.2" \
    "silero-vad>=5.1.2" \
    "pandas>=2.2.3" \
    "pyyaml>=6.0.2" \
    "aiofiles>=24.1.0" \
    "loguru>=0.7.3" \
    "transformers>=4.48.0"

echo ""
echo "=== Step 5: Install WhisperX v3.8.5 (--no-deps keeps Colab torch intact) ==="
pip install --quiet --no-deps \
    "git+https://github.com/m-bain/whisperX.git@v3.8.5"

echo ""
echo "=== Sanity check ==="
python - <<'PYEOF'
import sys
errors = []

try:
    import torch
    print(f"  ✅ torch          {torch.__version__}  | CUDA: {torch.cuda.is_available()}")
except Exception as e:
    errors.append(f"torch: {e}")

try:
    import numpy as np
    print(f"  ✅ numpy          {np.__version__}")
except Exception as e:
    errors.append(f"numpy: {e}")

try:
    import faster_whisper
    print(f"  ✅ faster-whisper {faster_whisper.__version__}")
except Exception as e:
    errors.append(f"faster-whisper: {e}")

try:
    import whisperx
    print(f"  ✅ whisperx       {whisperx.__version__}")
except Exception as e:
    errors.append(f"whisperx: {e}")

try:
    import fastapi
    print(f"  ✅ fastapi        {fastapi.__version__}")
except Exception as e:
    errors.append(f"fastapi: {e}")

try:
    import pyannote.audio
    print(f"  ✅ pyannote.audio {pyannote.audio.__version__}")
except Exception as e:
    errors.append(f"pyannote.audio: {e}")

if errors:
    print("\n❌ Issues found:")
    for err in errors:
        print(f"   {err}")
    sys.exit(1)
else:
    print("\n✅ All packages OK — ready to run!")
PYEOF

## 4 · Configuration — `config.yaml` & Environment Variables

**Where does each setting go?**

| Setting | Where to put it |
|---|---|
| Model size, batch size, chunk duration | `config.yaml` |
| HuggingFace token | `WHISPER_HF_TOKEN` env var (never commit to git) |
| API keys | `WHISPER_API_KEYS` env var |
| Redis URL | `REDIS_URL` env var |
| Device / compute type | `config.yaml` or `WHISPER_DEVICE` / `WHISPER_COMPUTE_TYPE` |

> All `WHISPER_*` env vars override the value in `config.yaml`.

In [ ]:
import os, torch

# ---------------------------------------------------------------------------
# Detect VRAM to auto-pick compute_type and batch_size
# ---------------------------------------------------------------------------
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    compute_type = "float16" if vram_gb >= 10 else "int8"
    batch_size   = 16 if vram_gb >= 10 else 8
    device       = "cuda"
    print(f"GPU detected: {torch.cuda.get_device_name(0)}  ({vram_gb:.1f} GB)")
    print(f"  → compute_type = {compute_type}")
    print(f"  → batch_size   = {batch_size}")
else:
    compute_type = "int8"
    batch_size   = 4
    device       = "cpu"
    print("No GPU — falling back to CPU / int8")

# ---------------------------------------------------------------------------
# Write config.yaml  (adjust values here as needed)
# ---------------------------------------------------------------------------
config_content = f"""# Auto-generated by Colab notebook
model:
  size: large-v2           # tiny | base | small | medium | large-v2 | large-v3
  device: {device}
  compute_type: {compute_type}
  download_root: /content/models   # Absolute path so weights persist on Colab VM

performance:
  batch_size: {batch_size}
  max_workers: 1           # Keep 1 on single-GPU Colab
  chunk_duration_sec: 30
  use_vad: true
  queue_soft_limit: 10000
  queue_hard_limit: null

accuracy:
  beam_size: 5
  temperature: 0.0
  best_of: 5

diarization:
  enabled: true
  hf_token: ""             # Set WHISPER_HF_TOKEN env var instead

webhook:
  enabled: true
  timeout_sec: 15
  retry_count: 3

security:
  api_key_enabled: true
  api_keys:
    - "colab-test-key"     # Set WHISPER_API_KEYS env var to override

paths:
  upload_dir: /content/uploads
  output_dir: /content/outputs
  temp_dir: /content/temp
"""

cfg_path = "/content/speech2text/config.yaml" if os.path.exists("/content/speech2text") else "config.yaml"
with open(cfg_path, "w") as f:
    f.write(config_content)
print(f"\nconfig.yaml written to: {cfg_path}")

# ---------------------------------------------------------------------------
# Validate by parsing with PyYAML
# ---------------------------------------------------------------------------
import yaml
with open(cfg_path) as f:
    parsed = yaml.safe_load(f)
print("\nParsed config (model section):", parsed["model"])
print("Parsed config (performance) :", parsed["performance"])

## 5 · Environment Variables

Set sensitive values here. They take priority over `config.yaml`.

In [ ]:
import os

# ============================================================
# ⚠️  FILL IN YOUR REAL VALUES HERE BEFORE RUNNING
# ============================================================

# Hugging Face token — required for pyannote diarization.
# Get yours at: https://huggingface.co/settings/tokens
# Accept model terms at: https://huggingface.co/pyannote/speaker-diarization-3.1
os.environ["WHISPER_HF_TOKEN"] = "hf_YOUR_TOKEN_HERE"   # <-- replace

# API key(s) — comma-separated list of valid keys
os.environ["WHISPER_API_KEYS"] = "colab-test-key"

# Redis (already started by colab_setup.sh / the bash cell above)
os.environ["REDIS_URL"] = "redis://localhost:6379/0"

# Optional: disable diarization if you don't have an HF token yet
# os.environ["WHISPER_DIARIZATION_ENABLED"] = "false"

# Optional: disable API key check for easier local testing
# os.environ["WHISPER_API_KEY_ENABLED"] = "false"

# ---------------------------------------------------------------------------
# Validate: print current effective env vars (redact secrets)
# ---------------------------------------------------------------------------
env_vars = [
    "REDIS_URL", "WHISPER_HF_TOKEN", "WHISPER_API_KEYS",
    "WHISPER_DEVICE", "WHISPER_COMPUTE_TYPE", "WHISPER_MODEL_SIZE",
    "WHISPER_DIARIZATION_ENABLED", "WHISPER_API_KEY_ENABLED",
]
print("Environment variable status:")
for k in env_vars:
    v = os.environ.get(k, "(not set)")
    # Redact long secrets
    display = v[:8] + "…" if len(v) > 12 and "TOKEN" in k else v
    status  = "✅" if v != "(not set)" else "⬜"
    print(f"  {status} {k:<35} = {display}")

## 6 · Webhook Test Server (ngrok tunnel)

We start a tiny FastAPI webhook receiver inside Colab and expose it via **pyngrok** to get a public HTTPS URL that the Speech-to-Text API can POST results to.

In [ ]:
%%bash
pip install --quiet pyngrok fastapi uvicorn

In [ ]:
import json, threading, os
from pathlib import Path
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
import uvicorn
from pyngrok import ngrok

WEBHOOK_LOG = Path("/content/webhook_payloads.jsonl")
WEBHOOK_PORT = 4000

# ---------------------------------------------------------------------------
# Tiny webhook receiver FastAPI app
# ---------------------------------------------------------------------------
webhook_app = FastAPI()
received_payloads = []

@webhook_app.post("/webhook")
async def receive_webhook(request: Request):
    payload = await request.json()
    received_payloads.append(payload)
    # Append to log file (one JSON object per line)
    with open(WEBHOOK_LOG, "a") as f:
        f.write(json.dumps(payload) + "\n")
    job_id = payload.get("job_id", "?")
    status = payload.get("status", "?")
    print(f"📩 Webhook received — job_id={job_id}  status={status}")
    if status == "completed":
        text = payload.get("result", {}).get("text", "")
        print(f"   text preview: {text[:120]}")
    return JSONResponse({"received": True})

@webhook_app.get("/health")
async def health():
    return {"status": "ok", "received": len(received_payloads)}

# ---------------------------------------------------------------------------
# Start server in background thread
# ---------------------------------------------------------------------------
def _run():
    uvicorn.run(webhook_app, host="0.0.0.0", port=WEBHOOK_PORT, log_level="warning")

t = threading.Thread(target=_run, daemon=True)
t.start()

# ---------------------------------------------------------------------------
# Open ngrok tunnel
# ---------------------------------------------------------------------------
# Optional: set your ngrok auth token to avoid session limits
# ngrok.set_auth_token("YOUR_NGROK_TOKEN_HERE")

tunnel = ngrok.connect(WEBHOOK_PORT)
WEBHOOK_PUBLIC_URL = tunnel.public_url + "/webhook"

os.environ["WEBHOOK_TEST_URL"] = WEBHOOK_PUBLIC_URL
print(f"\n✅ Webhook server running")
print(f"   Local  : http://localhost:{WEBHOOK_PORT}/webhook")
print(f"   Public : {WEBHOOK_PUBLIC_URL}")
print(f"\n   Use this URL as webhook_url when submitting jobs.")

## 7 · Start the API Server & Worker

In [ ]:
import subprocess, time, os

# Resolve repo root
REPO = "/content/speech2text" if os.path.exists("/content/speech2text") else os.getcwd()
print(f"Repo root: {REPO}")

env = os.environ.copy()
env["PYTHONPATH"] = REPO

# ---------------------------------------------------------------------------
# Start FastAPI (uvicorn) in background
# ---------------------------------------------------------------------------
api_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "app.main:app",
     "--host", "0.0.0.0", "--port", "8000",
     "--log-level", "info"],
    cwd=REPO,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(f"✅ API server started (PID {api_proc.pid}) on http://localhost:8000")

# ---------------------------------------------------------------------------
# Start RQ worker in background
# ---------------------------------------------------------------------------
worker_proc = subprocess.Popen(
    ["python", "-m", "rq", "worker", "transcription",
     "--url", os.environ.get("REDIS_URL", "redis://localhost:6379/0")],
    cwd=REPO,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(f"✅ RQ worker started (PID {worker_proc.pid})")

# Give services a moment to initialise
time.sleep(5)

# Health check
import urllib.request, json as _json
try:
    with urllib.request.urlopen("http://localhost:8000/health", timeout=5) as r:
        body = _json.loads(r.read())
    print(f"\n🟢 Health check: {body}")
except Exception as e:
    print(f"\n🔴 Health check failed: {e} — check API logs below")
    # Print first 40 lines of API stdout for debugging
    import select
    lines = []
    while True:
        ready = select.select([api_proc.stdout], [], [], 0.1)[0]
        if not ready:
            break
        line = api_proc.stdout.readline()
        if not line:
            break
        lines.append(line.decode())
    print("".join(lines[:40]))

## 8 · End-to-End Test

In [ ]:
import requests, time, json, os

API_BASE    = "http://localhost:8000"
API_KEY     = os.environ.get("WHISPER_API_KEYS", "colab-test-key").split(",")[0].strip()
WEBHOOK_URL = os.environ.get("WEBHOOK_TEST_URL", "")   # set by cell 6
AUDIO_URL   = "https://download.samplelib.com/mp3/sample-3s.mp3"

HEADERS = {"x-api-key": API_KEY}

# ---------------------------------------------------------------------------
# 1. Submit job
# ---------------------------------------------------------------------------
print("=== Submitting transcription job ===")
resp = requests.post(
    f"{API_BASE}/transcribe",
    data={
        "audio_url"  : AUDIO_URL,
        "metadata"   : json.dumps({"source": "colab_notebook_test"}),
        "webhook_url": WEBHOOK_URL,
    },
    headers=HEADERS,
    timeout=30,
)
resp.raise_for_status()
job = resp.json()
job_id = job["job_id"]
print(f"Job submitted — id: {job_id}  status: {job['status']}")

# ---------------------------------------------------------------------------
# 2. Poll status until done
# ---------------------------------------------------------------------------
print("\n=== Polling status ===")
MAX_WAIT = 600   # seconds
deadline = time.time() + MAX_WAIT
last_progress = -1

while time.time() < deadline:
    r = requests.get(f"{API_BASE}/status/{job_id}", headers=HEADERS, timeout=10)
    r.raise_for_status()
    state = r.json()
    status   = state["status"]
    progress = state.get("progress", 0)

    if progress != last_progress:
        print(f"  status={status:<12}  progress={progress}%")
        last_progress = progress

    if status == "completed":
        break
    if status == "failed":
        print(f"❌ Job failed: {state.get('error')}")
        break

    time.sleep(3)
else:
    print(f"⏱️ Timed out after {MAX_WAIT}s")

# ---------------------------------------------------------------------------
# 3. Fetch result
# ---------------------------------------------------------------------------
if status == "completed":
    print("\n=== Fetching result ===")
    r = requests.get(f"{API_BASE}/result/{job_id}", headers=HEADERS, timeout=30)
    r.raise_for_status()
    result = r.json().get("result", {})

    print(f"  language : {result.get('language', '?')}")
    print(f"  segments : {len(result.get('segments', []))}")
    print(f"  words    : {len(result.get('words', []))}")
    print(f"\n  Full text:\n  {result.get('text', '(empty)')}")

# ---------------------------------------------------------------------------
# 4. Show webhook payloads received
# ---------------------------------------------------------------------------
print("\n=== Webhook payloads received ===")
log_path = "/content/webhook_payloads.jsonl"
if os.path.exists(log_path):
    with open(log_path) as f:
        lines = f.readlines()
    print(f"  {len(lines)} webhook(s) received")
    for i, line in enumerate(lines, 1):
        p = json.loads(line)
        print(f"  [{i}] job_id={p.get('job_id')}  status={p.get('status')}")
else:
    if WEBHOOK_URL:
        print("  No log file yet — webhook may still be in transit")
    else:
        print("  (no webhook URL was set)")

## 9 · Cleanup (optional)

Run this cell to stop the API and worker processes when you are done.

In [ ]:
import os, signal

# Stop API server
try:
    api_proc.terminate()
    print(f"API server stopped (PID {api_proc.pid})")
except Exception as e:
    print(f"API stop: {e}")

# Stop RQ worker
try:
    worker_proc.terminate()
    print(f"RQ worker stopped (PID {worker_proc.pid})")
except Exception as e:
    print(f"Worker stop: {e}")

# Kill any leftover uvicorn processes
os.system("pkill -f 'uvicorn app.main'")
os.system("pkill -f 'rq worker'")

# Close ngrok tunnel
try:
    from pyngrok import ngrok as _ngrok
    _ngrok.disconnect(tunnel.public_url)
    print("ngrok tunnel closed")
except Exception:
    pass

print("Done.")